# Langfuse — RAG / LLM 可观测性实战

> **Langfuse** 是开源 LLM 工程平台，提供 Trace、Prompt 管理、评估与 Metrics，适合 RAG 上线后的调试与监控。

## 本 Notebook 覆盖

| Part | 主题 | 要点 |
|------|------|------|
| 1 | 环境配置 | API Key、连接校验 |
| 2 | 基础 Tracing | LangChain CallbackHandler |
| 3 | 进阶 Tracing | Trace ID、Session、@observe |
| 4 | Logging | Span 层级与日志级别 |
| 5 | Prompt 管理 | 从 Langfuse 拉取 Prompt |
| 6 | Metrics | 按 Trace 名称聚合统计 |

## 前置要求

在 `.env` 中配置（或通过环境变量）：

```env
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_HOST=https://cloud.langfuse.com   # 或自托管地址
OPENAI_API_KEY=sk-...
```

---

## Part 1 — 环境配置

加载环境变量并验证 Langfuse 客户端能否正常认证。`auth_check()` 返回 `True` 表示 Key 有效、可写入 Trace。

In [3]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)

True

In [4]:
from langfuse import get_client
 
langfuse_cli = get_client()
print(langfuse_cli.auth_check())

True


---

## Part 2 — 基础 Tracing（LangChain 集成）

Langfuse 通过 **`CallbackHandler`** 接入 LangChain：每次 `llm.invoke` / Chain 运行会自动上报：

- 输入 Prompt、输出 Completion
- Token 用量、延迟、模型名
- 嵌套的 Run 树（Chain → LLM → …）

在 [Langfuse Dashboard](https://cloud.langfuse.com) → **Traces** 中可查看刚产生的记录。

### 2A · 单次 LLM 调用

In [5]:
from langfuse.langchain import CallbackHandler
from langchain_openai import ChatOpenAI


langfuse_handler = CallbackHandler()
llm = ChatOpenAI(model_name="gpt-4.1-nano", callbacks=[langfuse_handler])
llm.invoke("return a number between 20 and 100")


AIMessage(content="Certainly! Here's a number between 20 and 100: 57.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 16, 'total_tokens': 31, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7f8eb7d1f9', 'id': 'chatcmpl-CwlhnOUmINe79J0HytvHX3QekT4Ii', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bac4f-4ca0-72e2-92ab-0a6c70037611-0', usage_metadata={'input_tokens': 16, 'output_tokens': 15, 'total_tokens': 31, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### 2B · LangChain Chain 链路追踪

Prompt 模板 + LLM 的 `RunnableSequence` 会产生**多层 Trace**（外层 Chain、内层 ChatOpenAI）。

通过 `config={"callbacks": [langfuse_handler]}` 传入 handler；Langfuse 会展示完整调用树。

In [6]:
from langchain_core.prompts import ChatPromptTemplate
 

llm = ChatOpenAI(model_name="gpt-4.1-nano")
prompt = ChatPromptTemplate.from_template("Tell me a short sentence about {topic}")
chain = prompt | llm
response = chain.invoke(
    {"topic": "cats"}, 
    config={"callbacks": [langfuse_handler]})
response

AIMessage(content='Cats are curious and independent animals.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 14, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7f8eb7d1f9', 'id': 'chatcmpl-CwlkM5JR7RMVWJb7WK9639TceqDw3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bac51-be0c-7121-b4f6-987c7d8bff0a-0', usage_metadata={'input_tokens': 14, 'output_tokens': 7, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

---

## Part 3 — 进阶 Tracing

生产环境需要把多次 LLM 调用**关联到同一条业务链路**：

| 概念 | 作用 |
|------|------|
| **trace_id** | 唯一标识一次用户请求 / 订单 |
| **session_id** | 多轮对话或同一用户会话 |
| **user_id / tags / metadata** | 筛选、分群、审计 |

`propagate_attributes()` 会把 session、user 等属性**传播**到当前 Trace 下所有子 Observation。

### 3A · 自定义 Trace + 属性传播

In [12]:
from langfuse import Langfuse, get_client, propagate_attributes
from langfuse.langchain import CallbackHandler
 
langfuse_cli = get_client()
langfuse_handler = CallbackHandler()
llm = ChatOpenAI(model_name="gpt-4.1-nano", callbacks=[langfuse_handler])

session_id = "session_666788"
order_id = "order_123457788"
trace_id = Langfuse.create_trace_id(seed=order_id)


with langfuse_cli.start_as_current_observation(
    as_type="span",
    name="langchain-call",
    trace_context={"trace_id": trace_id}
):
    # Propagate session_id to all observations
    with propagate_attributes(
        session_id=session_id,
        user_id="user001",
        tags=["api_01", "number"],
        metadata={"customer_level":"gold"}
    ):
        # Pass handler to the chain invocation
        resp = llm.invoke("return a number between 1 and 10")
        print(resp)

content='7' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1, 'prompt_tokens': 16, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7f8eb7d1f9', 'id': 'chatcmpl-Cwm0YEbEd3IlA6Hc0ZwK1wvPsk1fs', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019bac61-0f8c-74e0-9060-42105970ba7c-0' usage_metadata={'input_tokens': 16, 'output_tokens': 1, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


### 3B · 确定性 Trace ID

`Langfuse.create_trace_id(seed=order_id)` 可根据业务 ID（如订单号）生成**稳定、可复现**的 trace_id。

用途：同一订单重试时关联到同一条 Trace，或从业务日志反查 Langfuse 记录。

In [9]:
from langfuse import Langfuse

trace_id = Langfuse.create_trace_id()
trace_id

'0f51ff7fa89d52c157712caa23c236d1'

### 3C · `@observe` 装饰器

一行装饰即可把普通 Python 函数纳入 Trace，无需手动创建 Span。

适合包装：检索、Rerank、工具调用等非 LangChain 逻辑，与 LangChain Trace **出现在同一 Trace 树**（需在同一 trace 上下文中）。

In [50]:
from langfuse import observe
import random
 
@observe()
def my_observe_function():
    return random.randint(1, 10)

my_observe_function()

7

---

## Part 4 — Span 日志与级别

除 LLM 自动 Trace 外，可用 **`@observe`** 装饰器或 `start_as_current_observation()` 手动创建 Span。

- `update_current_span(level=..., status_message=...)` 写入 **DEBUG / DEFAULT / WARNING / ERROR**
- 嵌套 Span 形成调用树，便于定位 RAG 中「检索慢 / 生成错」等环节

> 在 Dashboard 中点击 Trace → 展开 Span 树，可看到各级别日志。

In [ ]:
from langfuse import observe, get_client
 
@observe(name="my-function-span")
def my_function():
    langfuse = get_client()
    for i in range(8):
        if i % 4 == 0:
            level="DEBUG"
        elif i % 3 == 1:
            level="DEFAULT"
        elif i % 3 == 2:
            level="WARNING"
        else:
            level="ERROR"
        
        with langfuse.start_as_current_observation(
            as_type="span",
            name=f"loop-{i}"
        ):
            langfuse_cli.update_current_span(
                status_message = f"This is a {level.lower()} ({i})",
                level = level
            )


my_function()

DEBUG
DEFAULT
WARNING
ERROR
DEBUG
WARNING
ERROR
DEFAULT


### 4B · 手动标记 Span 状态

在 LLM 调用前用 `update_current_span(level="ERROR", ...)` 标记业务事件（如「触发 LLM 请求」）。

可用于告警规则：Dashboard 或 Webhook 对 ERROR 级别 Span 触发通知。

In [13]:
from langfuse import Langfuse, get_client, propagate_attributes
from langfuse.langchain import CallbackHandler
 
langfuse_cli = get_client()
langfuse_handler = CallbackHandler()
llm = ChatOpenAI(model_name="gpt-4.1-nano", callbacks=[langfuse_handler])

session_id = "session_666"
order_id = "order_123457890000"
trace_id = Langfuse.create_trace_id(seed=order_id)


with langfuse_cli.start_as_current_observation(
    as_type="span",
    name="langchain-call2",
    # trace_context={"trace_id": trace_id}
) as obs:
    # Propagate session_id to all observations
    with propagate_attributes(
        session_id=session_id,
        user_id="user001",
        tags=["api_01", "number"],
        metadata={"customer_level":"gold"}
    ):
        langfuse_cli.update_current_span(
            status_message = "trigger llm request",
            level = "ERROR"
        )
        # Pass handler to the chain invocation
        resp = llm.invoke("return a number between 1 and 10")

---

## Part 5 — Prompt 管理

Langfuse 支持在控制台**版本化管理 Prompt**，代码侧通过名称拉取，实现：

- Prompt 与代码解耦，运营可独立迭代
- 按版本 A/B 对比效果
- 与 Trace 关联，复盘某次回答用了哪版 Prompt

下方从 Langfuse 读取名为 `my_prompt` 的 Prompt，并转为 LangChain 模板格式。

In [ ]:
from langfuse import get_client
 

langfuse_cli = get_client()
langfuse_prompt = langfuse_cli.get_prompt("my_prompt")
langfuse_prompt.get_langchain_prompt()

'This is a simple custom prompt stored on langfuse'

---

## Part 6 — Metrics 查询

Langfuse Metrics API 支持对 Trace / Observation 做**聚合分析**（类似简易 OLAP）：

- **measure**：`count`（次数）、`latency`（延迟）等
- **aggregation**：`sum` / `avg` / `count` / `p50` / `p90` / `p99` …
- **dimensions**：按 `name`、用户、标签等分组

下方查询指定时间窗口内、按 Trace `name` 分组的调用次数。可用于监控「某 RAG 接口 QPS」或对比不同 Chain 的使用量。

In [24]:
import json
from langfuse import get_client
 
 
langfuse_cli = get_client()
query = """
{
  "view": "traces",
  "metrics": [{"measure": "count", "aggregation": "count"}],
  "dimensions": [{"field": "name"}],
  "filters": [],
  "fromTimestamp": "2026-01-01T00:00:00Z",
  "toTimestamp": "2026-01-09T00:00:00Z"
}
"""
 # measure: count/latency
 # aggregation: sum/avg/count/max/min/p50/p75/p90/p99
rst = langfuse_cli.api.metrics.metrics(query = query)

print(json.dumps(rst.data, indent=4, ensure_ascii=False))

[
    {
        "name": "langchain-call",
        "count_count": "18"
    },
    {
        "name": "langchain-call2",
        "count_count": "11"
    },
    {
        "name": "my-function-span",
        "count_count": "8"
    },
    {
        "name": "my_function",
        "count_count": "5"
    },
    {
        "name": "ChatOpenAI",
        "count_count": "5"
    },
    {
        "name": "my_observe_function",
        "count_count": "5"
    },
    {
        "name": "RunnableSequence",
        "count_count": "3"
    },
    {
        "name": "loop-4",
        "count_count": "1"
    },
    {
        "name": "loop-0",
        "count_count": "1"
    },
    {
        "name": "loop-2",
        "count_count": "1"
    },
    {
        "name": "loop-3",
        "count_count": "1"
    },
    {
        "name": "loop-1",
        "count_count": "1"
    }
]
